### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [2]:
%pip install numpy scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [4]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [17]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [18]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [19]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [20]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [21]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [22]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [23]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [24]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


---
# Resolución:

In [25]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone
from sklearn.metrics import classification_report, accuracy_score

target_names = newsgroups_train.target_names

def mostrar(texto, n=400):
    """Muestra los primeros n caracteres de un documento, compactando espacios."""
    texto = ' '.join(texto.split())
    return texto[:n] + ('...' if len(texto) > n else '')

## 1. Similaridad entre documentos

Se toman 5 documentos al azar del conjunto de entrenamiento (con semilla fija para que el análisis sea reproducible). Cada uno se compara con todos los demás usando similaridad coseno sobre la vectorización TF-IDF (`X_train`) y se listan sus 5 vecinos más cercanos. El documento en sí queda excluido porque su similaridad consigo mismo es 1.

In [26]:
rng = np.random.default_rng(14)
docs_azar = rng.choice(X_train.shape[0], size=5, replace=False)

def vecinos_documento(idx, k=5):
    sims = cosine_similarity(X_train[idx], X_train)[0]
    top = np.argsort(sims)[::-1]
    top = top[top != idx][:k]
    return top, sims[top]

for idx in docs_azar:
    top, sims = vecinos_documento(idx)
    print('=' * 100)
    print(f'DOCUMENTO {idx} | clase: {target_names[y_train[idx]]}')
    print(mostrar(newsgroups_train.data[idx]))
    print('-' * 100)
    coincidencias = sum(y_train[j] == y_train[idx] for j in top)
    for j, s in zip(top, sims):
        marca = 'OK' if y_train[j] == y_train[idx] else 'X'
        print(f'  {marca} doc {j:5d} | sim={s:.3f} | {target_names[y_train[j]]:<25} | {mostrar(newsgroups_train.data[j], 110)}')
    print(f'  -> {coincidencias}/5 vecinos comparten la etiqueta')

DOCUMENTO 1003 | clase: comp.graphics
Hi all! I've just recently become seriously hooked on POV, but there are a few thing that I want to do that POV won't do (penumbral shadows, dispersion etc.). I was just wondering: what other shareware/freeware raytracers are out there, and what can they do? I've heard of Vivid and Polyray and Rayshade and so on, but I'd rather no wade through several hundred pages of manual for each trying to wor...
----------------------------------------------------------------------------------------------------
  OK doc  2468 | sim=0.204 | comp.graphics             | Yes, there are serveral programs which can convert font files (eq the Borland fonts) to objects consisting of ...
  OK doc  6404 | sim=0.188 | comp.graphics             | Yes, please create the group alt.raytrace soon!! I'm hooked on pov. geez. like I don't have anything better to...
  OK doc  5905 | sim=0.179 | comp.graphics             | Hallo POV-Renderers ! I've got a BocaX3 Card. Now I try to

### Interpretación

- **Doc 1003 (`comp.graphics`):** es una post pidiendo recomendacion sobre software de procesamiento grafico. Cuatro de los cinco vecinos son del mismo grupo y del mismo foro, por lo cual la similitud tiene sentido en su tematica y contenido. El quinto (`doc  4271 | talk.politics.misc`) es una transcripcion de un discurso de funcionarios del gobierno de EEUU, el texto es largo por lo que las palabras claves se destacan menos y tiende a convertirse en un texto generico en ingles. Esta puede ser la razon por la que equivocadamente se asocia al documento seleccionado.
- **Doc 1699 (`rec.sport.baseball`):** es una discusion sobre si algunos jugadores de baseball son judios o no. Los 5 documentos similares tratan el mismo tema y parecieran que son parte de la misma discusión. La similitud se da por palabras como judio y otras pertenecientes al vocubulario del baseball.
- **Doc 7390 (`sci.crypt`):** Este documento esta claisifcado como de la tematica de criptografia, sin embargo, es un debate desde un punto de vista politico sobre el derecho a una criptografia robusta. Esta mezcla de politicas para la criptografia asi como el uso de la metafora de arma para referirse a la criptografia, genera la similitudes con documentos de `talk.politics.guns` y `sci.crypt`.
- **Doc 9399 (`talk.politics.guns`):** es una discusion sobre la viabilidad de que Alemania invadiera Suiza durante la segunda guerra mundial, comentando que los suizos al portar armas armarian milicias civiles que defendiarian facilmente su territorio. Menciona terminos armas y politica. Genera similitudes con otros 5 documentos que parecen ser de la misma conversación y sobre portación de armas por civiles.
- **Doc 4083 (`sci.space`):** es una discusión sobre rayos gamas y agujeros negros. Este encuentra familiridad correctamente en otros 2 documentos sobre `sci.space` y rayos gamas. Un texto sobre `comp.windows.x` usa la palabra inglesa `holes`  para hacer referencia a vulnerbailidades de seguridad y por eso encuentra similitud con `black holes`. Los otros 2 documentos son largos por los que las palabran se diluyen y se termina convirtiendo en un texto ingles generico.

**Conclusión:** en total, 19 de los 25 vecinos comparten la etiqueta. Por lo general la similaridad coseno sobre TF-IDF recupera documentos del mismo tema, y muchas veces del mismo hilo de discusión, porque los textos citados repiten palabras raras. Sin embargo, a veces se confunde cuando una palabra se usa con otro significado como `guns` para referirse a criptografia robusta o el `holes` para amenzas de ciberseguridad. También, se confunde cuando los textos son largos y con tantas palabras que ninguna destaca.

## 2. Clasificador por prototipos (vecino más cercano, tipo *zero-shot*)

Cada documento de test se compara con **todos** los documentos de entrenamiento y recibe la etiqueta del documento de train con mayor similaridad coseno, es decir un *1-nearest-neighbour* con métrica coseno. No hay entrenamiento de parámetros: el "modelo" son los propios documentos etiquetados.

Se usa el mismo vectorizador TF-IDF ajustado en train. Como los vectores TF-IDF ya están normalizados, el coseno equivale al producto interno. Se procesa en bloques para no armar de una sola vez la matriz densa de 7532 × 11314.

In [27]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

def clasificar_por_prototipos(X_consulta, X_ref, y_ref, bloque=1000):
    preds = np.empty(X_consulta.shape[0], dtype=y_ref.dtype)
    for ini in range(0, X_consulta.shape[0], bloque):
        sims = cosine_similarity(X_consulta[ini:ini + bloque], X_ref)
        preds[ini:ini + bloque] = y_ref[sims.argmax(axis=1)]
    return preds

y_pred_proto = clasificar_por_prototipos(X_test, X_train, y_train)

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f'F1-score macro (prototipos): {f1_proto:.4f}')
print(f'Accuracy (prototipos):       {accuracy_score(y_test, y_pred_proto):.4f}')

f1_NB_base = f1_score(y_test, clf.predict(X_test), average='macro')
print(f'MultinomialNB base:          {f1_NB_base:.4f}')

F1-score macro (prototipos): 0.5050
Accuracy (prototipos):       0.5089
MultinomialNB base:          0.5854


### Interpretación

El clasificador por prototipos obtiene un **F1 macro de alrededor de 0.50**. Es mucho mejor que el azar (≈0.05 con 20 clases) y eso confirma lo visto en el punto 1: el vecino más similar suele ser de la misma clase. Pero no es mejor que un clasificar con Naive Bayes. Hay varias razones:

- **Depende de un solo documento.** Basta con que un documento de otra clase comparta un par de términos raros, como un saludo, una firma o una cita, para que la predicción salga mal. No hay ningún "promedio" sobre la clase que reduzca ese ruido.
- **Documentos vacíos o casi vacíos.** Al quitar headers, footers y quotes, más de 200 documentos de test quedan sin ningún término del vocabulario. Su vector es nulo, la similaridad con todos es 0 y `argmax` devuelve siempre el primer documento de train: la predicción es arbitraria.
- **Clases solapadas.** Existen clases cuyos documentos son muy parecidos por tener vocubulario similar, por ejemplo religión con ateísmo y cristianismo, política general con armas y Medio Oriente, electrónica con hardware.

Aun así, el método no necesita entrenamiento y sirve como línea de base. Se podría mejorar votando entre los *k* vecinos más cercanos, o comparando contra el **centroide** de cada clase en lugar de contra cada documento.

## 3. Naïve Bayes: maximizar el F1 macro

Para no "sobreajustar al test", los hiperparámetros se eligen con **validación cruzada de 5 folds sobre el conjunto de entrenamiento** (`GridSearchCV` con `scoring='f1_macro'`). El conjunto de test se usa **una sola vez** al final, para reportar el desempeño del mejor modelo. El vectorizador va dentro de un `Pipeline` para que en cada fold el vocabulario y el IDF se ajusten solo con los datos de entrenamiento de ese fold.

Se exploran (**sin tocar `ngram_range`**):

| Componente | Parámetro | Valores | Motivación |
|---|---|---|---|
| Modelo | clase | `MultinomialNB`, `ComplementNB` | Comparar ambas variantes |
| Modelo | `alpha` | 0.01, 0.1, 0.3, 1.0 | Intensidad del suavizado de Laplace |
| Vectorizador | `sublinear_tf` | `False`, `True` | Usar `1 + log(tf)` para que una palabra repetida muchas veces no domine el documento |

El vectorizador usa además `max_df=0.5`, fijo: descarta los términos presentes en más de la mitad de los documentos. En una búsqueda exploratoria más grande (160 combinaciones) también se probaron `stop_words='english'` y `min_df=2`. El primero no cambiaba el resultado (el IDF ya le da poco peso a esas palabras) y el segundo empeoraba levemente, así que se dejaron con sus valores por defecto para que la búsqueda sea más rápida.

In [28]:
pipe = Pipeline([
    ('vect', TfidfVectorizer(max_df=0.5)),
    ('clf', MultinomialNB()),
])

param_grid = {
    'clf': [MultinomialNB(), ComplementNB()],
    'clf__alpha': [0.01, 0.1, 0.3, 1.0],
    'vect__sublinear_tf': [False, True],
}

grid = GridSearchCV(pipe, param_grid, scoring='f1_macro', cv=5, n_jobs=-1)
grid.fit(newsgroups_train.data, y_train)

resultados = pd.DataFrame(grid.cv_results_)
resultados['modelo'] = resultados['param_clf'].apply(lambda m: type(m).__name__)
columnas = ['modelo', 'param_clf__alpha', 'param_vect__sublinear_tf', 'mean_test_score', 'std_test_score']
resultados[columnas].sort_values('mean_test_score', ascending=False).head(10)

,modelo,param_clf__alpha,param_vect__sublinear_tf,mean_test_score,std_test_score
13,ComplementNB,0.30,True,0.766332,0.008557
10,ComplementNB,0.10,False,0.764367,0.007975
12,ComplementNB,0.30,False,0.763800,0.008264
11,ComplementNB,0.10,True,0.763596,0.009762
0,MultinomialNB,0.01,False,0.757411,0.008555
14,ComplementNB,1.00,False,0.752330,0.009760
15,ComplementNB,1.00,True,0.751584,0.010792
1,MultinomialNB,0.01,True,0.751473,0.008902
8,ComplementNB,0.01,False,0.740545,0.006019
9,ComplementNB,0.01,True,0.736662,0.006233


Mejor configuración de **cada** modelo según la validación cruzada, y cómo cambia el F1 de validación con `alpha`:

In [29]:
mejor_por_modelo = resultados.loc[resultados.groupby('modelo')['mean_test_score'].idxmax(), columnas]
display(mejor_por_modelo)

,modelo,param_clf__alpha,param_vect__sublinear_tf,mean_test_score,std_test_score
13,ComplementNB,0.30,True,0.766332,0.008557
0,MultinomialNB,0.01,False,0.757411,0.008555


### Evaluación final en test

Se reentrenan con **todo** el conjunto de entrenamiento los dos modelos optimizados (el mejor MultinomialNB y el mejor ComplementNB según la validación cruzada) y se evalúan **una sola vez** en test. 

In [30]:
# Predicciones en test de los 4 modelos
predicciones = {
    'Prototipos (1-NN coseno)': y_pred_proto,
    'MultinomialNB sin ajuste': clf.predict(X_test),
}

for _, fila in mejor_por_modelo.iterrows():
    params = resultados.loc[fila.name, 'params']
    modelo = clone(pipe).set_params(**params).fit(newsgroups_train.data, y_train)
    nombre = f"{fila['modelo']} optimizado"
    predicciones[nombre] = modelo.predict(newsgroups_test.data)
    print(nombre, {k: v for k, v in params.items() if k != 'clf'})

comparacion = pd.DataFrame({
    nombre: {
        'F1 macro': f1_score(y_test, pred, average='macro'),
        'F1 ponderado': f1_score(y_test, pred, average='weighted'),
        'Accuracy': accuracy_score(y_test, pred),
    }
    for nombre, pred in predicciones.items()
}).T

base = comparacion.loc['MultinomialNB sin ajuste', 'F1 macro']
comparacion['Mejora F1 macro vs NB sin ajuste'] = comparacion['F1 macro'] - base
comparacion.sort_values('F1 macro', ascending=False).round(4)

ComplementNB optimizado {'clf__alpha': 0.3, 'vect__sublinear_tf': True}
MultinomialNB optimizado {'clf__alpha': 0.01, 'vect__sublinear_tf': False}


,F1 macro,F1 ponderado,Accuracy,Mejora F1 macro vs NB sin ajuste
ComplementNB optimizado,0.6987,0.7131,0.7183,0.1133
MultinomialNB optimizado,0.6829,0.6944,0.7001,0.0975
MultinomialNB sin ajuste,0.5854,0.6054,0.6062,0.0000
Prototipos (1-NN coseno),0.5050,0.5129,0.5089,-0.0804


### Comparación de los modelos en test

| Modelo | F1 macro | Accuracy | Mejora vs NB sin ajuste |
|---|---|---|---|
| ComplementNB optimizado | **0.699** | 0.718 | +0.113 |
| MultinomialNB optimizado | 0.683 | 0.700 | +0.098 |
| MultinomialNB sin ajuste | 0.585 | 0.606 | — |
| Prototipos (1-NN coseno) | 0.505 | 0.509 | −0.080 |

- **Los dos Naive Bayes optimizados superan claramente a las líneas de base.** El mejor modelo, ComplementNB, mejora 0.11 de F1 macro sobre el NB sin ajuste y 0.19 sobre prototipo. Toda la ganancia viene de elegir bien el modelo y sus hiperparámetros, sobre todo `alpha`.
- **El NB sin ajuste pierde por exceso de suavizado.** Con `alpha=1` el peso ficticio domina sobre los datos y las probabilidades de cada clase se vuelven parecidas entre sí. El MultinomialNB optimizado es el mismo modelo, con `alpha=0.01` y `max_df=0.5`, y recupera casi 0.10 de F1 macro. Casi toda esa mejora viene de `alpha`: en la búsqueda exploratoria, `max_df` casi no cambiaba el resultado.
- **Prototipos es el peor de los cuatro.** Decide con **un solo** documento de entrenamiento, así que cualquier coincidencia de palabras con un documento de otra clase produce un error. Naïve Bayes, en cambio, resume cada clase con la información de todos sus documentos, y por eso es más robusto. Además es mucho más rápido: no necesita comparar cada documento de test contra los 11 314 de train.
- **ComplementNB vs MultinomialNB optimizados.** La diferencia es chica (0.016 de F1 macro) pero consistente con la validación cruzada, donde ComplementNB también fue el mejor. Al estimar cada clase con los documentos de las otras 19, sus estimaciones usan más datos y son más estables.
- **F1 macro vs accuracy.** En todos los modelos el F1 macro es algo menor que la accuracy. El F1 macro le da el mismo peso a cada clase, así que lo bajan las clases difíciles, en las que todos los modelos rinden peor. La diferencia es chica porque las clases están bastante balanceadas.

In [31]:
mejor_modelo = grid.best_estimator_
y_pred_best = mejor_modelo.predict(newsgroups_test.data)
print('Mejor modelo según CV:', type(mejor_modelo.named_steps['clf']).__name__)
print(f"F1 macro test: {f1_score(y_test, y_pred_best, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred_best, target_names=target_names, digits=3))

Mejor modelo según CV: ComplementNB
F1 macro test: 0.6987

                          precision    recall  f1-score   support

             alt.atheism      0.324     0.436     0.372       319
           comp.graphics      0.730     0.717     0.724       389
 comp.os.ms-windows.misc      0.735     0.548     0.628       394
comp.sys.ibm.pc.hardware      0.636     0.714     0.673       392
   comp.sys.mac.hardware      0.782     0.727     0.754       385
          comp.windows.x      0.802     0.800     0.801       395
            misc.forsale      0.753     0.733     0.743       390
               rec.autos      0.813     0.747     0.779       396
         rec.motorcycles      0.817     0.774     0.795       398
      rec.sport.baseball      0.909     0.851     0.879       397
        rec.sport.hockey      0.875     0.945     0.908       399
               sci.crypt      0.755     0.811     0.782       396
         sci.electronics      0.716     0.570     0.635       393
                

### Interpretación

- **El ajuste de hiperparámetros sube mucho el desempeño.** El F1 macro en test pasa de **0.585** (MultinomialNB con valores por defecto) a **≈0.70** con ComplementNB optimizado. MultinomialNB optimizado también mejora mucho (≈0.68).
- **`alpha` es el hiperparámetro más importante, y su valor óptimo depende del modelo.** La clave es comparar el peso ficticio total ($\alpha \cdot V$) con el peso real de los datos. Con TF-IDF cada documento está normalizado y sus valores son chicos: una clase entera suma unas **4 000** unidades de peso, mientras que con `alpha=1` el suavizado suma $1 \cdot 100\,000$, **24 veces más** que los datos. Por eso MultinomialNB necesita un `alpha` muy chico (0.01) y empeora en forma sostenida al aumentarlo (F1 en CV: 0.757 → 0.644). ComplementNB, en cambio, estima cada clase con los documentos de las **otras 19** (unas 79 000 unidades de peso), así que tolera un suavizado mucho mayor: su óptimo está en 0.1–0.3, y con 0.01 empeora porque queda demasiado sensible a palabras raras.
- **ComplementNB supera a MultinomialNB.** ComplementNB estima los parámetros de cada clase con los documentos de **todas las otras clases** (su complemento). Así cada estimación usa muchos más datos y los pesos quedan menos sesgados cuando las clases tienen tamaños distintos. Por eso se recomienda especialmente para clasificación de texto.
- **`sublinear_tf` casi no cambia el resultado** (diferencias menores a 0.006 en CV). Ayuda levemente a ComplementNB y perjudica levemente a MultinomialNB.
- **Brecha entre validación y test (≈0.77 vs ≈0.70).** En 20 Newsgroups, el split de test está formado por mensajes **posteriores en el tiempo** a los de train, así que hay un cambio de distribución: hilos y temas nuevos. Por eso el F1 de validación cruzada es optimista, y es la razón para no elegir hiperparámetros mirando test.
- **Errores que persisten.** Las clases más difíciles siguen siendo `talk.religion.misc` (F1 ≈ 0.22), `alt.atheism` (≈ 0.37) y `talk.politics.misc` (≈ 0.53), casi las mismas que en el punto 2. Por ejemplo, `soc.religion.christian` tiene recall ≈ 0.90 pero precisión ≈ 0.55: "absorbe" muchos mensajes de `talk.religion.misc`, cuyo recall cae a ≈ 0.14, porque ambas hablan de Dios, la Biblia y Jesús. El problema es la representación BoW: si dos clases usan las mismas palabras, ningún modelo sobre BoW puede separarlas bien.

## 4. Similaridad entre palabras: matriz término-documento

Al transponer la matriz documento-término `X_train` se obtiene una matriz **término-documento**. En ella, cada palabra queda representada por un vector de 11.314 dimensiones, con su peso TF-IDF en cada documento del corpus. Dos palabras tendrán vectores parecidos si **aparecen en los mismos documentos**. Es decir, una palabra se conoce a partir de en que documento es usada.

Las palabras se eligieron **manualmente**, una por grupo temático bien diferenciado: `christian`, `politic`, `space`, `gun` e `game`.

In [32]:
X_terminos = X_train.T.tocsr()   # (n_terminos, n_documentos)
print(f'Matriz término-documento: {X_terminos.shape}')

def palabras_similares(palabra, matriz, vocab, idx2w, k=5):
    i = vocab[palabra]
    sims = cosine_similarity(matriz[i], matriz)[0]
    top = np.argsort(sims)[::-1]
    top = top[top != i][:k]
    return [(idx2w[j], round(float(sims[j]), 3)) for j in top]

palabras = ['christian', 'politic', 'space', 'gun', 'game']

tabla = {p: [f'{w} ({s})' for w, s in palabras_similares(p, X_terminos, tfidfvect.vocabulary_, idx2word)]
         for p in palabras}
pd.DataFrame(tabla, index=[f'#{i}' for i in range(1, 6)]).T

Matriz término-documento: (101631, 11314)


,#1,#2,#3,#4,#5
christian,christianity (0.234),christ (0.224),supremist (0.211),favourably (0.211),christians (0.211)
politic,disolving (0.968),complicitous (0.968),rectification (0.94),brendan (0.795),polity (0.732)
space,nasa (0.33),seds (0.297),shuttle (0.293),enfant (0.28),seti (0.246)
gun,guns (0.358),crime (0.244),handgun (0.239),homicides (0.233),firearms (0.233)
game,games (0.212),espn (0.194),hockey (0.181),team (0.179),scored (0.171)


### Interpretación

Los resultados son, en general, muy coherentes semánticamente:

- **christian** → `christianity`, `christ`, `christians`: comparten la raiz de la palabra. Las otras dos pueden aparecer muy frecuente en los textos tipicos de la religion cristiana.
- **politic** → `polity` comparte raiz con la palabra y `disolving`, `complicitous`, `rectification` son palabras muy usuales en textos politicos. La palabra `brendan` es la mas dificil de explicar, quizas es un nombre propio de algun politico o alto funcionario.
- **space** → `nasa`, `shuttle`, `seds` (*Students for the Exploration and Development of Space*), `seti`: todo vocabulario de `sci.space`.
- **gun** → `guns`, `handgun`, `firearms`, `crime`, `homicides`: la forma plural, sinónimos, y los temas del debate sobre control de armas.
- **game** → `games`, `espn`, `hockey`, `team`, `scored`: el plural y el resto todas palabras relacionados a los documentos de deportes.

Hay algunas observaciones importantes sobre esta representación:

1. **Captura relaciones temáticas, pero no sinonimos.** `gun` se parece a `crime` no porque signifiquen lo mismo sino porque aparecen en los mismos mensajes. 
2. **Misma palabra, difrente cantidad.** los singulares y plurales (`christian`/`christians`, `gun`/`guns`, `game`/`games`) aparecen porque no hay stemming ni lematización, así que el modelo no sabe que son la misma palabra.
3. **Las palabras poco frecuentes generan vecinos espurios.** Algunos vecinos de `space` como `enfant` y tambien puede ser el caso de `brendan`, aparecen en pocos documentos. Es decir, el vector de una palabra rara tiene muy pocos valores distintos de cero. Si dos palabras raras aparecen en los mismos documentos, sus vectores casi coinciden y el coseno sale muy alto. El TF-IDF no lo evita: el IDF les da más peso a las palabras raras, no menos.